# Συγκριτική Εκπαίδευση Κβαντικού Ταξινομητή (VQC) - COBYLA vs SPSA

**Dataset:** NSL-KDD (PCA-16) | **Κωδικοποίηση:** Amplitude Encoding (4 Qubits)

Το παρόν notebook υλοποιεί την εκπαίδευση ενός Variational Quantum Classifier (VQC) για τον διαχωρισμό της δικτυακής κίνησης σε Κανονική (Normal) και Κακόβουλη (Attack). Βασικός στόχος είναι η συγκριτική αξιολόγηση δύο διαφορετικών αλγορίθμων βελτιστοποίησης: 
1. **COBYLA** (Gradient-free, γρήγορος αλλά επιρρεπής σε τοπικά ελάχιστα).
2. **SPSA** (Στοχαστικός, ανθεκτικός στον θόρυβο, ιδανικός για κβαντικά τοπία).

---
## 1. Εισαγωγή Βιβλιοθηκών & Ρυθμίσεις

In [2]:
import os
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

# Scikit-Learn
from sklearn.preprocessing import normalize
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Qiskit Machine Learning & Algorithms
from qiskit.circuit.library import RealAmplitudes
from qiskit_machine_learning.circuit.library import raw_feature_vector
from qiskit_machine_learning.algorithms import VQC
from qiskit_algorithms.optimizers import SPSA, COBYLA
from qiskit.primitives import StatevectorSampler

warnings.filterwarnings('ignore')

## 2. Φόρτωση και Μετασχηματισμός Δεδομένων
Τα δεδομένα έχουν ήδη υποστεί Target Encoding και μείωση διαστάσεων (PCA) στα 16 χαρακτηριστικά. 
Για την ορθή εφαρμογή του **Amplitude Encoding**, είναι μαθηματικά υποχρεωτικό τα διανύσματα να υποστούν κανονικοποίηση **$L^2$** ($L^2$ normalization), ώστε το άθροισμα των τετραγώνων τους να ισούται με 1 ($||x||_2 = 1$). Παράλληλα, οι ετικέτες μετατρέπονται σε δυαδική μορφή (Normal $\rightarrow$ 0, Attack $\rightarrow$ 1).

In [3]:
print("--- Φόρτωση και Προετοιμασία Δεδομένων ---")
file_path = os.path.expanduser('~/Documents/projects/Quantum-IDS-NSLKDD/npz/nslkdd_pca_ready.npz')
data = np.load(file_path, allow_pickle=True)

# 1. Εξαγωγή δεδομένων με τα ΣΩΣΤΑ keys βάσει του .npz αρχείου σου
xTrain_raw, xTest_raw = data['xTrain_norm'], data['xTest_norm']
yTrain_full, yTest_full = data['yTrain_bin'], data['yTest_bin']

# 2. Κανονικοποίηση L2 (Απαραίτητη για Amplitude Encoding)
# Παίρνουμε τα PCA features και τα κανονικοποιούμε
xTrain_full = normalize(xTrain_raw, norm='l2', axis=1)
xTest_full = normalize(xTest_raw, norm='l2', axis=1)

# Τα yTrain_full και yTest_full είναι ήδη στο σωστό format (0/1),
# οπότε γλιτώνουμε το έξτρα binarization step εδώ!

print("Τα δεδομένα φορτώθηκαν και μετασχηματίστηκαν επιτυχώς.")
print(f"-> Διαστάσεις Συνόλου Εκπαίδευσης (Train): {xTrain_full.shape}")
print(f"-> Διαστάσεις Συνόλου Δοκιμής (Test):       {xTest_full.shape}")

--- Φόρτωση και Προετοιμασία Δεδομένων ---
Τα δεδομένα φορτώθηκαν και μετασχηματίστηκαν επιτυχώς.
-> Διαστάσεις Συνόλου Εκπαίδευσης (Train): (20155, 16)
-> Διαστάσεις Συνόλου Δοκιμής (Test):       (5037, 16)


## 3. Κατασκευή Κβαντικής Αρχιτεκτονικής & Εργαλείων
Το κύκλωμα αποτελείται από το `raw_feature_vector` (Amplitude Encoding) και το `RealAmplitudes` Ansatz (reps=5). 
Για να μην υπάρξει σύγκρουση μεταβλητών κατά τη σύγκριση, δημιουργείται μια συνάρτηση "εργοστάσιο" (`create_callback`) που παράγει ανεξάρτητα γραφήματα για κάθε optimizer.

In [4]:
NUM_QUBITS = 4
feature_map = raw_feature_vector(feature_dimension=16)
ansatz = RealAmplitudes(num_qubits=NUM_QUBITS, reps=5)

print(f"Κύκλωμα έτοιμο. Εκπαιδεύσιμες Παράμετροι: {ansatz.num_parameters}")

def create_callback(optimizer_name, color):
    """Επιστρέφει μια callback συνάρτηση και τη λίστα ιστορικού της για τον εκάστοτε optimizer"""
    history = []
    
    def callback(weights, obj_func_eval):
        clear_output(wait=True)
        history.append(obj_func_eval)
        
        plt.figure(figsize=(10, 5))
        plt.title(f"Εκπαίδευση VQC - {optimizer_name}", fontsize=14, pad=10)
        plt.plot(history, label="Loss", color=color, alpha=0.7)
        
        if len(history) > 10:
            ma = np.convolve(history, np.ones(10)/10, mode='valid')
            plt.plot(range(9, len(history)), ma, label="Trend (MA-10)", color="black", linestyle="--")
            
        plt.xlabel("Iterations", fontsize=11)
        plt.ylabel("Cross-Entropy Loss", fontsize=11)
        plt.legend()
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.show()
        print(f"[{optimizer_name}] Επανάληψη: {len(history)} | Loss: {obj_func_eval:.4f}")
        
    return callback, history

# Προετοιμασία φακέλου αποθήκευσης
save_folder = os.path.expanduser('~/Documents/projects/Quantum-IDS-NSLKDD/models/')
os.makedirs(save_folder, exist_ok=True)

Κύκλωμα έτοιμο. Εκπαιδεύσιμες Παράμετροι: 24


## 4. Πείραμα 1: Εκπαίδευση με COBYLA
Ο **COBYLA** είναι γρήγορος, αλλά επειδή λειτουργεί με τοπικές γραμμικές προσεγγίσεις, συχνά "παγιδεύεται" σε τοπικά ελάχιστα του κβαντικού χώρου, με αποτέλεσμα να σταματάει νωρίς τη βελτίωση του Loss.

In [ ]:
print("Εκκίνηση Εκπαίδευσης με COBYLA...")
cobyla_callback, cobyla_history = create_callback("COBYLA", "#1E88E5") # Μπλε χρώμα

vqc_cobyla = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=500),
    sampler=StatevectorSampler(),
    callback=cobyla_callback
)

start_time = time.time()
vqc_cobyla.fit(xTrain_full, yTrain_full)
print(f"\nCOBYLA Ολοκληρώθηκε σε {(time.time() - start_time) / 60:.2f} λεπτά.")

cobyla_path = os.path.join(save_folder, 'vqc_cobyla.model')
vqc_cobyla.to_dill(cobyla_path)

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


Εκκίνηση Εκπαίδευσης με COBYLA...


## 5. Πείραμα 2: Εκπαίδευση με SPSA
Ο **SPSA** απαιτεί περισσότερο χρόνο, αλλά η στοχαστική του φύση (εισαγωγή θορύβου στις παραμέτρους) του επιτρέπει να πλοηγείται αποτελεσματικά στα barren plateaus, επιτυγχάνοντας βαθύτερη σύγκλιση.

In [ ]:
print("Εκκίνηση Εκπαίδευσης με SPSA...")
spsa_callback, spsa_history = create_callback("SPSA", "#D81B60") # Κόκκινο/Φούξια χρώμα

vqc_spsa = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=SPSA(maxiter=500),
    sampler=StatevectorSampler(),
    callback=spsa_callback
)

start_time = time.time()
vqc_spsa.fit(xTrain_full, yTrain_full)
print(f"\nSPSA Ολοκληρώθηκε σε {(time.time() - start_time) / 60:.2f} λεπτά.")

spsa_path = os.path.join(save_folder, 'vqc_spsa.model')
vqc_spsa.to_dill(spsa_path)

No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.


--- 0. Άνοιγμα δεδομένων από: /home/aggelos/Documents/projects/Quantum-IDS-NSLKDD/npz/nslkdd_pca_ready.npz ---
✅ Δεδομένα έτοιμα (L2 Normalized).
🚀 Ξεκινάει ο Μαραθώνιος εκπαίδευσης σε 20155 δείγματα...


## 6. Συγκριτική Αξιολόγηση (COBYLA vs SPSA)
Πραγματοποιείται inference στο Test Set (5.037 εγγραφές) και για τα δύο μοντέλα. Εξάγονται οι μετρικές και σχεδιάζονται οι Πίνακες Σύγχυσης δίπλα-δίπλα για άμεση σύγκριση.

In [ ]:
save_folder = os.path.expanduser('~/Documents/projects/Quantum-IDS-NSLKDD/models/')
cobyla_path = os.path.join(save_folder, 'vqc_cobyla.model')
spsa_path = os.path.join(save_folder, '_vqc_spsa_final_88acc.model') # Το ακριβές όνομα που ζήτησες

print("Φόρτωση μοντέλων από τον δίσκο...")
try:
    vqc_cobyla = VQC.from_dill(cobyla_path)
    vqc_spsa = VQC.from_dill(spsa_path)
except AttributeError:
    # Σε περίπτωση που είχαν σωθεί με την παλιά μέθοδο .save()
    vqc_cobyla = VQC.load(cobyla_path)
    vqc_spsa = VQC.load(spsa_path)
print("Τα μοντέλα φορτώθηκαν επιτυχώς!\n")

# 2. Ο ΚΩΔΙΚΑΣ ΣΟΥ ΓΙΑ ΠΡΟΒΛΕΨΕΙΣ ΚΑΙ ΑΞΙΟΛΟΓΗΣΗ
def get_predictions(model, x_test):
    y_pred_raw = model.predict(x_test)
    if len(y_pred_raw.shape) > 1 and y_pred_raw.shape[1] > 1:
        return np.argmax(y_pred_raw, axis=1)
    return y_pred_raw

# Προβλέψεις (μπορεί να πάρει λίγο χρόνο για τα 5.000 δείγματα)
print("🚀 Εξαγωγή προβλέψεων στο Test Set...")
y_pred_cobyla = get_predictions(vqc_cobyla, xTest_full)
y_pred_spsa = get_predictions(vqc_spsa, xTest_full)

print("="*50)
print("REPORT COBYLA:")
print(classification_report(yTest_full, y_pred_cobyla, target_names=["Normal (0)", "Attack (1)"]))
print("="*50)
print("REPORT SPSA:")
print(classification_report(yTest_full, y_pred_spsa, target_names=["Normal (0)", "Attack (1)"]))

# Οπτικοποίηση δίπλα-δίπλα
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_cobyla = confusion_matrix(yTest_full, y_pred_cobyla)
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm_cobyla, display_labels=["Normal", "Attack"])
disp1.plot(cmap='Blues', ax=axes[0], values_format='d')
axes[0].set_title("Πίνακας Σύγχυσης - COBYLA")

cm_spsa = confusion_matrix(yTest_full, y_pred_spsa)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_spsa, display_labels=["Normal", "Attack"])
disp2.plot(cmap='Reds', ax=axes[1], values_format='d')
axes[1].set_title("Πίνακας Σύγχυσης - SPSA")

plt.tight_layout()
plt.savefig("vqc_optimizer_comparison.png", dpi=300)
print("Το γράφημα αποθηκεύτηκε ως 'vqc_optimizer_comparison.png'")
plt.show()

NameError: name 'os' is not defined